In [7]:
# [Cell 1] - Imports
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Embedding, LSTM, Dense, Input, Concatenate, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [8]:
# [Cell 2] - Load Data
data_path = r"C:\Users\ronle\Desktop\BUAS\Y2C\data_medium_preprocessed.csv"
df = pd.read_csv(data_path)

In [9]:
# [Cell 3] - Prepare Features and Labels
# Text processing
texts = df['text'].astype(str).tolist()
max_num_words = 10000
max_sequence_length = 100
embedding_dim = 100

# Tokenize text
tokenizer = Tokenizer(num_words=max_num_words, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
X_text = pad_sequences(sequences, maxlen=max_sequence_length, padding='post')

# Scale Sentiment Score
scaler = StandardScaler()
sentiment_scores = scaler.fit_transform(df[['Sentiment_Score']])

# Prepare labels
emotion_cols = ['happy', 'sad', 'disgusted', 'mad', 'scared', 'surprised', 'neutral']
labels = df[emotion_cols].values
y = np.argmax(labels, axis=1)
y_cat = to_categorical(y, num_classes=len(emotion_cols))

In [10]:
# [Cell 4] - Split Data
X_train_text, X_val_text, X_train_sentiment, X_val_sentiment, y_train, y_val = train_test_split(
    X_text,
    sentiment_scores,
    y_cat,
    test_size=0.2,
    random_state=42
)

In [11]:
# [Cell 5] - Build Model
# Text input branch
text_input = Input(shape=(max_sequence_length,), name='text_input')
embedding = Embedding(input_dim=min(max_num_words, len(tokenizer.word_index) + 1),
                     output_dim=embedding_dim,
                     input_length=max_sequence_length)(text_input)
lstm = LSTM(128, dropout=0.2, recurrent_dropout=0.1)(embedding)

# Sentiment score branch
sentiment_input = Input(shape=(1,), name='sentiment_input')
sentiment_dense = Dense(32, activation='relu')(sentiment_input)

# Combine branches
combined = Concatenate()([lstm, sentiment_dense])
dense1 = Dense(128, activation='relu')(combined)
dropout1 = Dropout(0.3)(dense1)
dense2 = Dense(64, activation='relu')(dropout1)
output = Dense(len(emotion_cols), activation='softmax')(dense2)

# Create and compile model
model = Model(inputs=[text_input, sentiment_input], outputs=output)
model.compile(loss='categorical_crossentropy', 
             optimizer='adam', 
             metrics=['accuracy'])

# Display model summary
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 text_input (InputLayer)        [(None, 100)]        0           []                               
                                                                                                  
 embedding (Embedding)          (None, 100, 100)     1000000     ['text_input[0][0]']             
                                                                                                  
 sentiment_input (InputLayer)   [(None, 1)]          0           []                               
                                                                                                  
 lstm (LSTM)                    (None, 128)          117248      ['embedding[0][0]']              
                                                                                              

In [12]:
# [Cell 6] - Train Model
epochs = 10
batch_size = 32

history = model.fit(
    [X_train_text, X_train_sentiment],
    y_train,
    validation_data=([X_val_text, X_val_sentiment], y_val),
    epochs=epochs,
    batch_size=batch_size
)

Epoch 1/10
4296/4296 [==============================] - 464s 107ms/step - loss: 1.3693 - accuracy: 0.4766 - val_loss: 1.3600 - val_accuracy: 0.4770
Epoch 2/10
4296/4296 [==============================] - 485s 113ms/step - loss: 1.3612 - accuracy: 0.4783 - val_loss: 1.3599 - val_accuracy: 0.4768
Epoch 3/10
4296/4296 [==============================] - 479s 111ms/step - loss: 1.3599 - accuracy: 0.4791 - val_loss: 1.3607 - val_accuracy: 0.4755
Epoch 4/10
4296/4296 [==============================] - 480s 112ms/step - loss: 1.3592 - accuracy: 0.4781 - val_loss: 1.3596 - val_accuracy: 0.4765
Epoch 5/10
4296/4296 [==============================] - 480s 112ms/step - loss: 1.3586 - accuracy: 0.4788 - val_loss: 1.3594 - val_accuracy: 0.4765
Epoch 6/10
4296/4296 [==============================] - 481s 112ms/step - loss: 1.3582 - accuracy: 0.4791 - val_loss: 1.3611 - val_accuracy: 0.4768
Epoch 7/10
4296/4296 [==============================] - 486s 113ms/step - loss: 1.3577 - accuracy: 0.4794 - val_

In [13]:
# Get predictions
y_pred = model.predict([X_val_text, X_val_sentiment])

# Convert predictions to class indices
y_true_classes = np.argmax(y_val, axis=1)
y_pred_classes = np.argmax(y_pred, axis=1)

# Calculate metrics
accuracy = accuracy_score(y_true_classes, y_pred_classes)
precision = precision_score(y_true_classes, y_pred_classes, average='weighted')
recall = recall_score(y_true_classes, y_pred_classes, average='weighted')
f1 = f1_score(y_true_classes, y_pred_classes, average='weighted')

# Print metrics
print("\nOverall Metrics:")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

1074/1074 [==============================] - 29s 26ms/step

Overall Metrics:
Accuracy: 0.4758
Precision: 0.3959
Recall: 0.4758
F1 Score: 0.4140


c:\Users\ronle\anaconda3\envs\y2c\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [14]:
# [Cell 8] - Save Model (Optional)
model.save("lstm_model_4.h5")